In [5]:
import json
import pandas as pd
import numpy as np

from docx import Document

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm

In [6]:
# --------------------------
# File Paths
# --------------------------

CANDIDATE_FILE = "../data/sample_candidates.json"

JOB_DESCRIPTION_FILE = "../data/job_description.docx"

OUTPUT_FILE = "../output/ranked_candidates.csv"

In [7]:
with open(CANDIDATE_FILE,"r",encoding="utf-8") as f:
    candidates = json.load(f)

print("Candidates Loaded:",len(candidates))

Candidates Loaded: 50


In [8]:
doc = Document(JOB_DESCRIPTION_FILE)

job_text = ""

for para in doc.paragraphs:
    job_text += para.text + "\n"

print(job_text[:700])

Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)

Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we're going to t


In [9]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Model Loaded Successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Loaded Successfully


In [10]:
job_embedding = model.encode(job_text)

print(job_embedding.shape)

(384,)


In [11]:
def get_skills(candidate):

    skills = []

    for skill in candidate["skills"]:

        skills.append(
            f"{skill['name']} ({skill['proficiency']}, {skill['duration_months']} months)"
        )

    return "\n".join(skills)

In [12]:
print(get_skills(candidates[0]))

Tailwind (intermediate, 13 months)
NLP (advanced, 26 months)
Image Classification (advanced, 40 months)
Fine-tuning LLMs (advanced, 36 months)
Weights & Biases (intermediate, 30 months)
Speech Recognition (advanced, 33 months)
Photoshop (intermediate, 24 months)
TTS (advanced, 60 months)
LoRA (intermediate, 28 months)
Apache Beam (intermediate, 9 months)
AWS (beginner, 8 months)
Flask (beginner, 15 months)
BentoML (intermediate, 36 months)
Milvus (advanced, 35 months)
GANs (advanced, 19 months)
Statistical Modeling (intermediate, 8 months)
GCP (beginner, 2 months)


In [13]:
def get_career(candidate):

    career = []

    for job in candidate["career_history"]:

        text = f"""
Role: {job['title']}
Company: {job['company']}
Industry: {job['industry']}
Duration: {job['duration_months']} months

Responsibilities:
{job['description']}
"""

        career.append(text)

    return "\n".join(career)

In [14]:
print(get_career(candidates[0]))


Role: Backend Engineer
Company: Mindtree
Industry: IT Services
Duration: 27 months

Responsibilities:
Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with some adjacent ML exposure.


Role: Analytics Engineer
Company: Dunder Mifflin
Industry: Paper Products
Duration: 55 months

Responsibilities:
Built and maintained data pipelines on Apache Airflow processing ~500GB of daily transactional data across 12 source systems. Worked extensively with Spark (PySpark) for batch processing and dbt for the transformation/modeling layer in our Snowflake warehouse. Owned the on-call rotation for data quality issues — wrote most of the data qu

In [15]:
def get_education(candidate):

    education = []

    for edu in candidate["education"]:

        education.append(
            f"""
Degree: {edu['degree']}
Field: {edu['field_of_study']}
Institution: {edu['institution']}
CGPA/Grade: {edu['grade']}
Years: {edu['start_year']} - {edu['end_year']}
"""
        )

    return "\n".join(education)

In [16]:
print(get_education(candidates[0]))


Degree: B.E.
Field: Computer Science
Institution: Lovely Professional University
CGPA/Grade: 8.24 CGPA
Years: 2017 - 2020



In [17]:
def get_certifications(candidate):

    if len(candidate["certifications"]) == 0:
        return "No Certifications"

    certs = []

    for cert in candidate["certifications"]:
        certs.append(str(cert))

    return "\n".join(certs)

In [18]:
print(get_certifications(candidates[0]))

No Certifications


In [19]:
def get_languages(candidate):

    languages = []

    for language in candidate["languages"]:

        languages.append(
            f"{language['language']} ({language['proficiency']})"
        )

    return ", ".join(languages)

In [20]:
print(get_languages(candidates[0]))

English (professional), Hindi (conversational)


In [21]:
def get_behavior(candidate):

    s = candidate["redrob_signals"]

    return f"""
Open To Work: {s['open_to_work_flag']}
GitHub Activity Score: {s['github_activity_score']}
Recruiter Response Rate: {s['recruiter_response_rate']}
Interview Completion Rate: {s['interview_completion_rate']}
Profile Completeness: {s['profile_completeness_score']}
Notice Period: {s['notice_period_days']} days
"""

In [22]:
print(get_behavior(candidates[0]))


Open To Work: True
GitHub Activity Score: 9.2
Recruiter Response Rate: 0.34
Interview Completion Rate: 0.71
Profile Completeness: 86.9
Notice Period: 60 days



In [23]:
def create_candidate_profile(candidate):

    profile = candidate["profile"]

    return f"""
==============================
CANDIDATE PROFILE
==============================

Current Role:
{profile['current_title']}

Current Company:
{profile['current_company']}

Industry:
{profile['current_industry']}

Years of Experience:
{profile['years_of_experience']}

Headline:
{profile['headline']}

Professional Summary:
{profile['summary']}

==============================
SKILLS
==============================

{get_skills(candidate)}

==============================
CAREER HISTORY
==============================

{get_career(candidate)}

==============================
EDUCATION
==============================

{get_education(candidate)}

==============================
CERTIFICATIONS
==============================

{get_certifications(candidate)}

==============================
LANGUAGES
==============================

{get_languages(candidate)}

==============================
BEHAVIORAL SIGNALS
==============================

{get_behavior(candidate)}
"""

In [24]:
candidate_profile = create_candidate_profile(candidates[0])

print(candidate_profile[:2500])


CANDIDATE PROFILE

Current Role:
Backend Engineer

Current Company:
Mindtree

Industry:
IT Services

Years of Experience:
6.9

Headline:
Backend Engineer | SQL, Spark, Cloud

Professional Summary:
Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.

SKILLS

Tailwind (intermediate, 13 months)
NLP (advanced, 26 months)
Image Classification (advanced, 40 months)
Fine-tuning LLMs (advanced, 36 months)
Weights & Biase

In [25]:
SKILL_ONTOLOGY = {

    "Python": [
        "Python"
    ],

    "Machine Learning": [
        "Machine Learning",
        "Scikit-learn",
        "TensorFlow",
        "PyTorch",
        "XGBoost",
        "LightGBM"
    ],

    "Deep Learning": [
        "CNN",
        "RNN",
        "LSTM",
        "GANs",
        "Transformers"
    ],

    "LLM": [
        "Fine-tuning LLMs",
        "LoRA",
        "PEFT",
        "Prompt Engineering"
    ],

    "NLP": [
        "NLP",
        "BERT",
        "RoBERTa",
        "T5",
        "spaCy"
    ],

    "Embeddings": [
        "Sentence Transformers",
        "Embeddings",
        "BGE",
        "E5"
    ],

    "Vector Database": [
        "Milvus",
        "FAISS",
        "Pinecone",
        "Qdrant",
        "Weaviate"
    ],

    "Cloud": [
        "AWS",
        "Azure",
        "GCP"
    ],

    "Big Data": [
        "Spark",
        "Kafka",
        "Airflow",
        "Apache Beam"
    ]
}

In [26]:
def normalize_candidate_skills(candidate):

    normalized = set()

    candidate_skills = [
        skill["name"]
        for skill in candidate["skills"]
    ]

    for category, skills in SKILL_ONTOLOGY.items():

        for skill in skills:

            if skill in candidate_skills:
                normalized.add(category)

    return normalized

In [27]:
normalize_candidate_skills(candidates[0])

{'Big Data', 'Cloud', 'Deep Learning', 'LLM', 'NLP', 'Vector Database'}

In [28]:
def extract_required_skills(job_text):

    required = set()

    text = job_text.lower()

    for category, skills in SKILL_ONTOLOGY.items():

        if category.lower() in text:
            required.add(category)

        for skill in skills:

            if skill.lower() in text:
                required.add(category)

    return sorted(list(required))

In [29]:
required_skills = extract_required_skills(job_text)

print(required_skills)

['Deep Learning', 'Embeddings', 'LLM', 'Machine Learning', 'NLP', 'Python', 'Vector Database']


In [30]:
def calculate_skill_score(candidate, required_skills):

    candidate_skills = normalize_candidate_skills(candidate)

    matched = sorted(
        list(
            candidate_skills.intersection(required_skills)
        )
    )

    score = len(matched) / max(len(required_skills), 1)

    return score, matched

In [31]:
skill_score, matched_skills = calculate_skill_score(
    candidates[0],
    required_skills
)

print("Skill Score:", skill_score)

print("Matched Skills:", matched_skills)

Skill Score: 0.5714285714285714
Matched Skills: ['Deep Learning', 'LLM', 'NLP', 'Vector Database']


In [32]:
AI_ROLES = [
    "AI Engineer",
    "Machine Learning Engineer",
    "ML Engineer",
    "Data Scientist",
    "Research Engineer",
    "Backend Engineer",
    "Data Engineer",
    "Analytics Engineer"
]

In [33]:
AI_TECHNOLOGIES = [

    "Python",
    "Spark",
    "Kafka",
    "Airflow",
    "PyTorch",
    "TensorFlow",
    "NLP",
    "Embeddings",
    "LoRA",
    "Milvus",
    "FAISS",
    "LLM",
    "AWS",
    "Azure",
    "GCP",
    "Vector Database",
    "Machine Learning",
    "Apache Beam"
]

In [34]:
def calculate_experience_score(candidate):

    score = 0

    # ---------- Years ----------
    years = candidate["profile"]["years_of_experience"]

    score += min(years / 10, 1) * 0.40

    # ---------- Relevant Roles ----------
    role_bonus = 0

    for job in candidate["career_history"]:

        if job["title"] in AI_ROLES:
            role_bonus += 0.15

    score += min(role_bonus, 0.30)

    # ---------- Technology Bonus ----------
    description = " ".join(
        job["description"]
        for job in candidate["career_history"]
    ).lower()

    tech_matches = 0

    for tech in AI_TECHNOLOGIES:

        if tech.lower() in description:
            tech_matches += 1

    tech_score = min(tech_matches / 10, 1) * 0.30

    score += tech_score

    return round(min(score, 1),4)

In [35]:
experience_score = calculate_experience_score(
    candidates[0]
)

print(experience_score)

0.666


In [36]:
def calculate_behavior_score(candidate):

    s = candidate["redrob_signals"]

    score = 0

    # Open to Work
    if s["open_to_work_flag"]:
        score += 0.20

    # GitHub Activity
    score += (s["github_activity_score"] / 100) * 0.20

    # Recruiter Response
    score += s["recruiter_response_rate"] * 0.20

    # Interview Completion
    score += s["interview_completion_rate"] * 0.20

    # Profile Completeness
    score += (s["profile_completeness_score"] / 100) * 0.20

    return round(score,4)

In [37]:
behavior_score = calculate_behavior_score(
    candidates[0]
)

print(behavior_score)

0.6022


In [38]:
def generate_reason(candidate,
                    semantic_score,
                    skill_score,
                    behavior_score,
                    experience_score,
                    matched_skills):

    reasons = []

    # Semantic Match
    if semantic_score >= 0.70:
        reasons.append("Excellent semantic match with the job description.")
    elif semantic_score >= 0.50:
        reasons.append("Good semantic match with the job description.")
    else:
        reasons.append("Moderate semantic match with the job description.")

    # Skill Match
    if len(matched_skills) > 0:
        reasons.append(
            "Matched skill categories: " +
            ", ".join(matched_skills)
        )

    # Experience
    years = candidate["profile"]["years_of_experience"]

    reasons.append(
        f"{years} years of professional experience."
    )

    # Open to Work
    if candidate["redrob_signals"]["open_to_work_flag"]:
        reasons.append(
            "Currently open to work."
        )

    # GitHub
    github = candidate["redrob_signals"]["github_activity_score"]

    if github >= 7:
        reasons.append(
            "Strong GitHub activity."
        )

    # Profile Completeness
    completeness = candidate["redrob_signals"]["profile_completeness_score"]

    if completeness >= 80:
        reasons.append(
            "Well-maintained professional profile."
        )

    return " ".join(reasons)

In [39]:
reason = generate_reason(
    candidates[0],
    semantic_score,
    skill_score,
    behavior_score,
    experience_score,
    matched_skills
)

print(reason)

NameError: name 'semantic_score' is not defined

In [40]:
candidate_profile = create_candidate_profile(candidates[0])

candidate_embedding = model.encode(candidate_profile)

In [41]:
semantic_score = cosine_similarity(
    [job_embedding],
    [candidate_embedding]
)[0][0]

print("Semantic Score:", round(semantic_score, 4))

Semantic Score: 0.5647


In [42]:
reason = generate_reason(
    candidates[0],
    semantic_score,
    skill_score,
    behavior_score,
    experience_score,
    matched_skills
)

print(reason)

Good semantic match with the job description. Matched skill categories: Deep Learning, LLM, NLP, Vector Database 6.9 years of professional experience. Currently open to work. Strong GitHub activity. Well-maintained professional profile.


In [43]:
def calculate_final_score(
    semantic_score,
    skill_score,
    experience_score,
    behavior_score
):

    weights = {
        "semantic": 0.35,
        "skills": 0.30,
        "experience": 0.20,
        "behavior": 0.15
    }

    score = (
        semantic_score * weights["semantic"] +
        skill_score * weights["skills"] +
        experience_score * weights["experience"] +
        behavior_score * weights["behavior"]
    )

    return round(score, 4)

In [44]:
final_score = calculate_final_score(
    semantic_score,
    skill_score,
    experience_score,
    behavior_score
)

print("Final Score:", final_score)

Final Score: 0.5926


In [45]:
results = []

for candidate in tqdm(candidates):

    # Build profile
    profile = create_candidate_profile(candidate)

    # Generate embedding
    candidate_embedding = model.encode(profile)

    # Semantic score
    semantic_score = cosine_similarity(
        [job_embedding],
        [candidate_embedding]
    )[0][0]

    # Skill score
    skill_score, matched_skills = calculate_skill_score(
        candidate,
        required_skills
    )

    # Experience
    experience_score = calculate_experience_score(candidate)

    # Behavior
    behavior_score = calculate_behavior_score(candidate)

    # Final score
    final_score = calculate_final_score(
        semantic_score,
        skill_score,
        experience_score,
        behavior_score
    )

    # Reason
    reason = generate_reason(
        candidate,
        semantic_score,
        skill_score,
        behavior_score,
        experience_score,
        matched_skills
    )

    results.append({
        "candidate_id": candidate["candidate_id"],
        "semantic_score": round(semantic_score, 4),
        "skill_score": round(skill_score, 4),
        "experience_score": round(experience_score, 4),
        "behavior_score": round(behavior_score, 4),
        "final_score": final_score,
        "reason": reason
    })

100%|██████████| 50/50 [00:05<00:00,  9.30it/s]


In [46]:
results_df = pd.DataFrame(results)

results_df.head()

,candidate_id,semantic_score,skill_score,experience_score,behavior_score,final_score,reason
0,CAND_0000001,0.5647,0.5714,0.666,0.6022,0.5926,Good semantic match with the job description. ...
1,CAND_0000002,0.5776,0.0000,0.430,0.5374,0.3688,Good semantic match with the job description. ...
2,CAND_0000003,0.5749,0.0000,0.044,0.3258,0.2589,Good semantic match with the job description. ...
3,CAND_0000004,0.5753,0.0000,0.182,0.1770,0.2643,Good semantic match with the job description. ...
4,CAND_0000005,0.5689,0.0000,0.400,0.5892,0.3675,Good semantic match with the job description. ...


In [47]:
results_df = results_df.sort_values(
    by="final_score",
    ascending=False
)

results_df.head(10)

,candidate_id,semantic_score,skill_score,experience_score,behavior_score,final_score,reason
0,CAND_0000001,0.5647,0.5714,0.666,0.6022,0.5926,Good semantic match with the job description. ...
20,CAND_0000021,0.6308,0.4286,0.400,0.3338,0.4794,Good semantic match with the job description. ...
30,CAND_0000031,0.4461,0.4286,0.240,0.7340,0.4428,Moderate semantic match with the job descripti...
9,CAND_0000010,0.5204,0.4286,0.334,0.4166,0.4400,Good semantic match with the job description. ...
13,CAND_0000014,0.6085,0.2857,0.336,0.3918,0.4247,Good semantic match with the job description. ...
14,CAND_0000015,0.5530,0.2857,0.276,0.5648,0.4192,Good semantic match with the job description. ...
31,CAND_0000032,0.5656,0.2857,0.354,0.3628,0.4089,Good semantic match with the job description. ...
37,CAND_0000038,0.6182,0.1429,0.298,0.5372,0.3994,Good semantic match with the job description. ...
42,CAND_0000043,0.5516,0.2857,0.362,0.2640,0.3908,Good semantic match with the job description. ...
40,CAND_0000041,0.5809,0.0000,0.430,0.5338,0.3694,Good semantic match with the job description. ...


In [48]:
results_df.reset_index(drop=True, inplace=True)

results_df["rank"] = results_df.index + 1

results_df.head(10)

,candidate_id,semantic_score,skill_score,experience_score,behavior_score,final_score,reason,rank
0,CAND_0000001,0.5647,0.5714,0.666,0.6022,0.5926,Good semantic match with the job description. ...,1
1,CAND_0000021,0.6308,0.4286,0.400,0.3338,0.4794,Good semantic match with the job description. ...,2
2,CAND_0000031,0.4461,0.4286,0.240,0.7340,0.4428,Moderate semantic match with the job descripti...,3
3,CAND_0000010,0.5204,0.4286,0.334,0.4166,0.4400,Good semantic match with the job description. ...,4
4,CAND_0000014,0.6085,0.2857,0.336,0.3918,0.4247,Good semantic match with the job description. ...,5
5,CAND_0000015,0.5530,0.2857,0.276,0.5648,0.4192,Good semantic match with the job description. ...,6
6,CAND_0000032,0.5656,0.2857,0.354,0.3628,0.4089,Good semantic match with the job description. ...,7
7,CAND_0000038,0.6182,0.1429,0.298,0.5372,0.3994,Good semantic match with the job description. ...,8
8,CAND_0000043,0.5516,0.2857,0.362,0.2640,0.3908,Good semantic match with the job description. ...,9
9,CAND_0000041,0.5809,0.0000,0.430,0.5338,0.3694,Good semantic match with the job description. ...,10


In [49]:
results_df.to_csv(
    "../output/ranked_candidates.csv",
    index=False
)

print("CSV Saved Successfully!")

CSV Saved Successfully!


In [50]:
results_df[
    [
        "rank",
        "candidate_id",
        "final_score",
        "reason"
    ]
].head(10)

,rank,candidate_id,final_score,reason
0,1,CAND_0000001,0.5926,Good semantic match with the job description. ...
1,2,CAND_0000021,0.4794,Good semantic match with the job description. ...
2,3,CAND_0000031,0.4428,Moderate semantic match with the job descripti...
3,4,CAND_0000010,0.4400,Good semantic match with the job description. ...
4,5,CAND_0000014,0.4247,Good semantic match with the job description. ...
5,6,CAND_0000015,0.4192,Good semantic match with the job description. ...
6,7,CAND_0000032,0.4089,Good semantic match with the job description. ...
7,8,CAND_0000038,0.3994,Good semantic match with the job description. ...
8,9,CAND_0000043,0.3908,Good semantic match with the job description. ...
9,10,CAND_0000041,0.3694,Good semantic match with the job description. ...


In [51]:
top_candidates = results_df.head(10)

print(top_candidates[["candidate_id", "final_score"]])

   candidate_id  final_score
0  CAND_0000001       0.5926
1  CAND_0000021       0.4794
2  CAND_0000031       0.4428
3  CAND_0000010       0.4400
4  CAND_0000014       0.4247
5  CAND_0000015       0.4192
6  CAND_0000032       0.4089
7  CAND_0000038       0.3994
8  CAND_0000043       0.3908
9  CAND_0000041       0.3694


In [52]:
prompt = f"""
You are an experienced technical recruiter.

Job Description:
{job_text}

Candidate Profile:
{candidate_profile}

Current AI Score:
{final_score}

Evaluate this candidate.

Return ONLY:

Overall Match Score (0-100)

Strengths

Weaknesses

Hiring Recommendation
"""

In [53]:
def create_llm_prompt(job_text, candidate_profile, final_score):

    prompt = f"""
You are a Senior Technical Recruiter with over 15 years of hiring experience.

Your task is to evaluate whether the candidate is a good fit for the job.

=========================
JOB DESCRIPTION
=========================

{job_text}

=========================
CANDIDATE PROFILE
=========================

{candidate_profile}

=========================
CURRENT AI MATCH SCORE
=========================

{round(final_score * 100, 2)}%

=========================
INSTRUCTIONS
=========================

Evaluate the candidate holistically.

Consider:

1. Career History
2. Technical Skills
3. Relevant Experience
4. Education
5. Industry Background
6. Behavioral Signals
7. Overall Fit

Return ONLY the following format.

Overall Match Score: XX/100

Strengths:
- ...
- ...
- ...

Weaknesses:
- ...
- ...

Hiring Recommendation:
Highly Recommended / Recommended / Consider / Not Recommended

Reason:
Write 3-4 concise sentences explaining your decision.

"""
    return prompt

In [54]:
prompt = create_llm_prompt(
    job_text,
    candidate_profile,
    final_score
)

print(prompt)


You are a Senior Technical Recruiter with over 15 years of hiring experience.

Your task is to evaluate whether the candidate is a good fit for the job.

JOB DESCRIPTION

Job Description: Senior AI Engineer — Founding Team
Company: Redrob AI (Series A AI-native talent intelligence platform)
Location: Pune/Noida, India (Hybrid — flexible cadence) | Open to relocation candidates from Tier-1 Indian cities
Employment Type: Full-time
Experience Required: 5–9 years (see "what we mean by this" below)

Let's be honest about this role
We're going to write this JD differently from most. We're a Series A company that just raised our round and we're building a new AI Engineering org from scratch. This is the kind of role where the JD changes every six months because the company changes every six months. So instead of pretending we have a fixed checklist, we're going to tell you what we actually need and what we've gotten wrong before.
If you've spent your career at Google or Meta and you want a w

In [55]:
results.append({
    "candidate_id": candidate["candidate_id"],
    "semantic_score": round(semantic_score, 4),
    "skill_score": round(skill_score, 4),
    "experience_score": round(experience_score, 4),
    "behavior_score": round(behavior_score, 4),
    "matched_skills": ", ".join(matched_skills),
    "final_score": round(final_score, 4),
    "reason": reason
})

In [56]:
def get_recommendation(score):

    if score >= 0.75:
        return "Highly Recommended"

    elif score >= 0.60:
        return "Recommended"

    elif score >= 0.45:
        return "Consider"

    else:
        return "Not Recommended"

In [57]:
results_df["recommendation"] = results_df["final_score"].apply(get_recommendation)